In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
DATA_PATH = Path("../data/raw/creditcard.csv")

df = pd.read_csv(DATA_PATH)
df = df.sort_values("Time").reset_index(drop=True)

train_end = int(len(df) * 0.60)
validation_end = int(len(df) * 0.80)

train_df = df.iloc[:train_end].copy()
validation_df = df.iloc[train_end:validation_end].copy()
test_df = df.iloc[validation_end:].copy()

FEATURES = [column for column in df.columns if column != "Class"]

X_train = train_df[FEATURES]
y_train = train_df["Class"]

X_validation = validation_df[FEATURES]
y_validation = validation_df["Class"]

X_test = test_df[FEATURES]
y_test = test_df["Class"]

print(f"Validation transactions: {len(validation_df):,}")
print(f"Validation fraud cases: {y_validation.sum():,}")

Validation transactions: 56,961
Validation fraud cases: 57


In [2]:
scale_features = ["Time", "Amount"]
pca_features = [f"V{i}" for i in range(1, 29)]

preprocessor = ColumnTransformer(
    transformers=[
        ("scaled", RobustScaler(), scale_features),
        ("pca", "passthrough", pca_features),
    ]
)

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

logistic_model.fit(X_train, y_train)

validation_scores = logistic_model.predict_proba(
    X_validation
)[:, 1]

print(
    "Validation average precision:",
    round(
        average_precision_score(
            y_validation,
            validation_scores,
        ),
        4,
    ),
)

Validation average precision: 0.7727


In [3]:
REVIEW_CAPACITY = 100

ranked_indices = np.argsort(validation_scores)[::-1]
review_indices = ranked_indices[:REVIEW_CAPACITY]

capacity_predictions = np.zeros(
    len(y_validation),
    dtype=int,
)
capacity_predictions[review_indices] = 1

capacity_threshold = validation_scores[review_indices[-1]]

print(f"Review capacity: {REVIEW_CAPACITY}")
print(f"Score threshold: {capacity_threshold:.6f}")
print(f"Transactions flagged: {capacity_predictions.sum()}")

Review capacity: 100
Score threshold: 0.986487
Transactions flagged: 100


In [4]:
tn, fp, fn, tp = confusion_matrix(
    y_validation,
    capacity_predictions,
).ravel()

precision = precision_score(
    y_validation,
    capacity_predictions,
)

recall = recall_score(
    y_validation,
    capacity_predictions,
)

print(f"True positives:  {tp}")
print(f"False positives: {fp}")
print(f"False negatives: {fn}")
print(f"True negatives:  {tn}")
print()
print(f"Precision: {precision:.2%}")
print(f"Recall:    {recall:.2%}")

True positives:  45
False positives: 55
False negatives: 12
True negatives:  56849

Precision: 45.00%
Recall:    78.95%


In [ ]:
def evaluate_capacity(y_true, scores, capacity):
    y_array = np.asarray(y_true)
    ranked = np.argsort(scores)[::-1]
    selected = ranked[:capacity]

    predictions = np.zeros(len(y_array), dtype=int)
    predictions[selected] = 1

    tn, fp, fn, tp = confusion_matrix(
        y_array,
        predictions,
    ).ravel()

    return {
        "review_capacity": capacity,
        "score_threshold": scores[selected[-1]],
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "precision": tp / (tp + fp),
        "recall": tp / (tp + fn),
        "alerts_per_1000": capacity / len(y_array) * 1000,
    }

In [ ]:
capacity_results = pd.DataFrame(
    [
        evaluate_capacity(
            y_validation,
            validation_scores,
            capacity,
        )
        for capacity in [25, 50, 100, 250, 500, 1000]
    ]
)

capacity_results

In [ ]:
capacity_display = capacity_results.copy()

capacity_display["precision"] = (
    capacity_display["precision"].map("{:.2%}".format)
)

capacity_display["recall"] = (
    capacity_display["recall"].map("{:.2%}".format)
)

capacity_display["score_threshold"] = (
    capacity_display["score_threshold"].map("{:.6f}".format)
)

capacity_display

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.plot(
    capacity_results["review_capacity"],
    capacity_results["recall"],
    marker="o",
    color="tab:blue",
    label="Recall",
)

ax1.set_xlabel("Transactions reviewed")
ax1.set_ylabel("Fraud recall", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()

ax2.plot(
    capacity_results["review_capacity"],
    capacity_results["precision"],
    marker="s",
    color="tab:orange",
    label="Precision",
)

ax2.set_ylabel("Alert precision", color="tab:orange")
ax2.tick_params(axis="y", labelcolor="tab:orange")

plt.title("Investigation Workload Trade-off")
plt.show()

In [ ]:
validation_analysis = validation_df.copy()

validation_analysis["risk_score"] = validation_scores
validation_analysis["flagged_for_review"] = capacity_predictions

validation_analysis["outcome"] = np.select(
    [
        (validation_analysis["Class"] == 1)
        & (validation_analysis["flagged_for_review"] == 1),

        (validation_analysis["Class"] == 0)
        & (validation_analysis["flagged_for_review"] == 1),

        (validation_analysis["Class"] == 1)
        & (validation_analysis["flagged_for_review"] == 0),
    ],
    [
        "True positive",
        "False positive",
        "False negative",
    ],
    default="True negative",
)

validation_analysis["outcome"].value_counts()

In [ ]:
false_negatives = (
    validation_analysis[
        validation_analysis["outcome"] == "False negative"
    ]
    .sort_values("risk_score", ascending=False)
)

false_negatives[
    [
        "Time",
        "Amount",
        "risk_score",
        "Class",
    ]
]

In [ ]:
false_negatives[
    ["Amount", "risk_score"]
].describe()

In [ ]:
false_positives = (
    validation_analysis[
        validation_analysis["outcome"] == "False positive"
    ]
    .sort_values("risk_score", ascending=False)
)

false_positives[
    [
        "Time",
        "Amount",
        "risk_score",
        "Class",
    ]
].head(20)

In [ ]:
outcome_amount_summary = (
    validation_analysis.groupby("outcome")["Amount"]
    .agg(["count", "mean", "median", "max"])
    .sort_values("count", ascending=False)
)

outcome_amount_summary

In [ ]:
processed_features = (
    scale_features + pca_features
)

coefficients = logistic_model.named_steps[
    "classifier"
].coef_[0]

coefficient_df = pd.DataFrame(
    {
        "feature": processed_features,
        "coefficient": coefficients,
        "absolute_coefficient": np.abs(coefficients),
    }
).sort_values(
    "absolute_coefficient",
    ascending=False,
)

coefficient_df.head(15)

In [ ]:
top_coefficients = coefficient_df.head(15).sort_values(
    "coefficient"
)

plt.figure(figsize=(9, 7))

colors = [
    "tab:red" if value < 0 else "tab:blue"
    for value in top_coefficients["coefficient"]
]

plt.barh(
    top_coefficients["feature"],
    top_coefficients["coefficient"],
    color=colors,
)

plt.axvline(0, color="black", linewidth=1)
plt.xlabel("Logistic-regression coefficient")
plt.ylabel("Feature")
plt.title("Largest Logistic-Regression Coefficients")
plt.show()

## Threshold selection and error analysis

At an illustrative review capacity of 100 transactions, logistic regression
detected 45 of 57 fraud cases. This produced 45% alert precision and 78.95%
fraud recall.

Increasing review capacity improved recall but reduced precision, demonstrating
the trade-off between fraud losses and investigation workload. The operating
point is therefore a business decision rather than a purely statistical one.

The default probability threshold of 0.5 generated substantially more alerts.
A capacity-based policy produced a more practical alert queue by ranking
transactions and reviewing only the highest-risk cases.

Feature interpretation is limited because V1–V28 are anonymous PCA components.
Coefficient direction and magnitude can describe model behaviour, but cannot be
translated into merchant, customer or device characteristics.